# 🚀 Fine-tune Embedding Model với Synthetic Training Data

**Mục tiêu:**
- Load corpus từ `dev_phase/corpus/` (allergy, cardiovascular, diabetes, respiratory)
- Tự động generate training queries từ documents
- Tạo positive/negative pairs cho contrastive learning
- Fine-tune embedding model (sentence-transformers)
- Evaluate performance: Before vs After
- Download fine-tuned model để integrate vào RAG pipeline

**Base Model:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384 dims)

**Training Method:** Contrastive Learning với CosineSimilarityLoss

**Output:** `./models/embedding_finetuned/` (ready for ChromaDB re-indexing)

In [ ]:
# Check GPU availability (important for training speed)
import torch

if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Training will be slower on CPU.")
    print("   Recommendation: Runtime → Change runtime type → T4 GPU")

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
# Install sentence-transformers for fine-tuning
!pip install -q sentence-transformers datasets scikit-learn

print("✅ Dependencies installed:")
print("   - sentence-transformers (fine-tuning framework)")
print("   - datasets (data handling)")
print("   - scikit-learn (evaluation metrics)")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os
import random
from typing import List, Tuple, Dict
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Sentence transformers
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from torch.utils.data import DataLoader

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

# Google Colab
from google.colab import files

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
import torch
torch.manual_seed(42)

print("✅ Libraries imported successfully")

## Bước 2: Upload Corpus Data

**Cần upload 4 files từ `E:\CĐTT2\WeatherWell AI\dev_phase\corpus\`:**
- `allergy_chunks.csv`
- `cardiovascular_chunks.csv`
- `diabetes_chunks.csv`
- `respiratory_chunks.csv`

**Total:** ~200-500 KB (text data)

In [ ]:
# Upload corpus files
print("📤 Uploading corpus files...")
print("   Please select all 4 *_chunks.csv files")
print("")

uploaded = files.upload()

print(f"\n✅ Uploaded {len(uploaded)} files:")
for filename in uploaded.keys():
    print(f"   - {filename} ({len(uploaded[filename])/1024:.1f} KB)")

## Bước 3: Load và merge corpus documents

In [ ]:
# Load all corpus files
corpus_files = [f for f in uploaded.keys() if f.endswith('_chunks.csv')]

all_documents = []
category_counts = {}

for file in corpus_files:
    df = pd.read_csv(file)
    category = file.replace('_chunks.csv', '')
    
    # Add category to each chunk
    for idx, row in df.iterrows():
        all_documents.append({
            'id': f"{category}_{idx}",
            'text': row['chunk'],
            'url': row.get('url', ''),
            'category': category
        })
    
    category_counts[category] = len(df)
    print(f"✅ Loaded {category}: {len(df)} chunks")

print(f"\n📊 Total corpus:")
print(f"   Documents: {len(all_documents)}")
print(f"   Categories: {len(category_counts)}")
print(f"   Distribution: {category_counts}")

# Create DataFrame
corpus_df = pd.DataFrame(all_documents)
print(f"\n📋 Sample documents:")
print(corpus_df.head())

## Bước 4: Generate Synthetic Queries từ Documents

**Phương pháp:**
- Extract key phrases từ documents
- Tạo question templates (how, what, when, why)
- Generate variations với paraphrasing
- Mỗi document → 3-5 queries

**Target:** 1000-2000 training pairs

In [ ]:
# Query generation templates
QUERY_TEMPLATES = [
    # Health condition questions
    "triệu chứng của {topic} là gì",
    "{topic} ảnh hưởng như thế nào",
    "cách phòng tránh {topic}",
    "điều trị {topic} như thế nào",
    "{topic} có nguy hiểm không",
    "nguyên nhân gây {topic}",
    "{topic} là gì",
    "ai dễ bị {topic}",
    
    # Weather-health relationship
    "thời tiết ảnh hưởng đến {topic}",
    "nhiệt độ cao có gây {topic}",
    "độ ẩm cao ảnh hưởng {topic}",
    "UV cao có gây {topic}",
    "ô nhiễm không khí và {topic}",
    "pm2.5 ảnh hưởng {topic}",
    
    # General health questions
    "khi nào cần đi bác sĩ cho {topic}",
    "biến chứng của {topic}",
    "thuốc điều trị {topic}",
    "chế độ ăn cho người {topic}",
    "tập thể dục có tốt cho {topic}"
]

def extract_key_phrases(text: str) -> List[str]:
    """
    Extract key health-related phrases from document
    Simple rule-based extraction
    """
    # Common health keywords
    health_keywords = [
        'dị ứng', 'allergy', 'allergies', 'asthma', 'hen suyễn',
        'tim mạch', 'cardiovascular', 'heart', 'huyết áp',
        'tiểu đường', 'diabetes', 'đường huyết', 'insulin',
        'hô hấp', 'respiratory', 'phổi', 'lung', 'COPD',
        'da', 'skin', 'mắt', 'eye', 'mũi', 'nose'
    ]
    
    text_lower = text.lower()
    found_topics = []
    
    for keyword in health_keywords:
        if keyword.lower() in text_lower:
            found_topics.append(keyword)
    
    return list(set(found_topics))[:3]  # Max 3 topics per doc

def generate_queries_for_document(doc: Dict, num_queries: int = 3) -> List[str]:
    """
    Generate synthetic queries for a document
    """
    text = doc['text']
    category = doc['category']
    
    # Extract topics
    topics = extract_key_phrases(text)
    
    # If no specific topics, use category
    if not topics:
        topics = [category]
    
    # Generate queries
    queries = []
    templates_sample = random.sample(QUERY_TEMPLATES, min(num_queries, len(QUERY_TEMPLATES)))
    
    for template in templates_sample:
        topic = random.choice(topics)
        query = template.format(topic=topic)
        queries.append(query)
    
    return queries

# Generate queries for all documents
print("🔄 Generating synthetic queries...")

training_pairs = []

for idx, doc in enumerate(all_documents):
    queries = generate_queries_for_document(doc, num_queries=3)
    
    for query in queries:
        training_pairs.append({
            'query': query,
            'positive_doc': doc['text'],
            'doc_id': doc['id'],
            'category': doc['category']
        })
    
    if (idx + 1) % 100 == 0:
        print(f"   Processed {idx + 1}/{len(all_documents)} documents...")

print(f"\n✅ Generated {len(training_pairs)} query-document pairs")
print(f"\n📋 Sample pairs:")
for i in range(3):
    pair = training_pairs[i]
    print(f"\nPair {i+1}:")
    print(f"  Query: {pair['query']}")
    print(f"  Doc: {pair['positive_doc'][:100]}...")
    print(f"  Category: {pair['category']}")

## Bước 5: Tạo Negative Samples

**Contrastive Learning cần:**
- ✅ Positive pairs: (query, relevant_doc) → similarity = 1.0
- ✅ Negative pairs: (query, irrelevant_doc) → similarity = 0.0

**Strategy:** Random sample documents từ khác category

In [ ]:
# Create negative samples
print("🔄 Creating negative samples...")

# Group documents by category
docs_by_category = defaultdict(list)
for doc in all_documents:
    docs_by_category[doc['category']].append(doc['text'])

# Add negative samples to each pair
for pair in training_pairs:
    current_category = pair['category']
    
    # Get documents from different categories
    other_categories = [cat for cat in docs_by_category.keys() if cat != current_category]
    
    if other_categories:
        # Random select category
        neg_category = random.choice(other_categories)
        # Random select document from that category
        pair['negative_doc'] = random.choice(docs_by_category[neg_category])
    else:
        # If only one category, random select different document
        all_texts = [d['text'] for d in all_documents if d['text'] != pair['positive_doc']]
        pair['negative_doc'] = random.choice(all_texts)

print(f"✅ Created negative samples for {len(training_pairs)} pairs")

# Create final training DataFrame
training_df = pd.DataFrame(training_pairs)

print(f"\n📊 Training data summary:")
print(f"   Total pairs: {len(training_df)}")
print(f"   Unique queries: {training_df['query'].nunique()}")
print(f"   Categories: {training_df['category'].value_counts().to_dict()}")

print(f"\n📋 Sample training pair:")
sample = training_df.iloc[0]
print(f"Query: {sample['query']}")
print(f"\nPositive (relevant): {sample['positive_doc'][:150]}...")
print(f"\nNegative (irrelevant): {sample['negative_doc'][:150]}...")

## Bước 6: Split Train/Validation

In [ ]:
# Split train/validation (80/20)
train_df, val_df = train_test_split(
    training_df,
    test_size=0.2,
    random_state=42,
    stratify=training_df['category']  # Maintain category distribution
)

print("📊 Data split:")
print(f"   Training: {len(train_df)} pairs ({len(train_df)/len(training_df)*100:.1f}%)")
print(f"   Validation: {len(val_df)} pairs ({len(val_df)/len(training_df)*100:.1f}%)")

print(f"\n📈 Category distribution:")
print(f"\nTraining:")
print(train_df['category'].value_counts())
print(f"\nValidation:")
print(val_df['category'].value_counts())

## Bước 7: Load Base Embedding Model

**Model:** `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Dimensions: 384
- Supports: 50+ languages (including Vietnamese)
- Size: ~470 MB
- Performance: Good balance speed/accuracy

In [ ]:
# Load pre-trained model
MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

print(f"📥 Loading base model: {MODEL_NAME}")
print("   This may take 1-2 minutes...")

base_model = SentenceTransformer(MODEL_NAME)

print(f"\n✅ Model loaded successfully!")
print(f"   Model: {base_model}")
print(f"   Embedding dimension: {base_model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length: {base_model.max_seq_length}")

# Test embedding
test_text = "thời tiết nóng ảnh hưởng đến sức khỏe"
test_embedding = base_model.encode(test_text)
print(f"\n🧪 Test embedding:")
print(f"   Input: {test_text}")
print(f"   Output shape: {test_embedding.shape}")
print(f"   Sample values: {test_embedding[:5]}")

## Bước 8: Chuẩn bị Training Examples

In [ ]:
# Create InputExample objects for sentence-transformers
from sentence_transformers import InputExample

print("🔄 Creating training examples...")

train_examples = []

for idx, row in train_df.iterrows():
    # Positive pair (query, positive_doc) with label 1.0
    train_examples.append(
        InputExample(texts=[row['query'], row['positive_doc']], label=1.0)
    )
    
    # Negative pair (query, negative_doc) with label 0.0
    train_examples.append(
        InputExample(texts=[row['query'], row['negative_doc']], label=0.0)
    )

print(f"✅ Created {len(train_examples)} training examples")
print(f"   Positive pairs: {len(train_examples)//2}")
print(f"   Negative pairs: {len(train_examples)//2}")

# Create validation examples
val_examples = []

for idx, row in val_df.iterrows():
    val_examples.append(
        InputExample(texts=[row['query'], row['positive_doc']], label=1.0)
    )
    val_examples.append(
        InputExample(texts=[row['query'], row['negative_doc']], label=0.0)
    )

print(f"\n✅ Created {len(val_examples)} validation examples")

# Create DataLoader
BATCH_SIZE = 16  # Adjust based on GPU memory
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

print(f"\n📦 DataLoader created:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total batches: {len(train_dataloader)}")

## Bước 9: Evaluate Base Model & Save Embeddings

In [ ]:
# CRITICAL: Save base model embeddings BEFORE fine-tuning
print("🔍 Evaluating base model (before fine-tuning)...")

# Prepare validation data
val_query_texts = [ex.texts[0] for ex in val_examples]
val_doc_texts = [ex.texts[1] for ex in val_examples]
val_labels = np.array([ex.label for ex in val_examples])

print(f"   Validation pairs: {len(val_query_texts)}")
print(f"   Positive samples: {sum(val_labels)}")
print(f"   Negative samples: {len(val_labels) - sum(val_labels)}")

# Encode with BASE model
print(f"\n   Encoding {len(val_query_texts)} validation pairs...")
base_query_embeddings = base_model.encode(val_query_texts, show_progress_bar=True)
base_doc_embeddings = base_model.encode(val_doc_texts, show_progress_bar=True)

# Calculate cosine similarities
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
base_similarities = []
for i in range(len(base_query_embeddings)):
    sim = cos_sim([base_query_embeddings[i]], [base_doc_embeddings[i]])[0][0]
    base_similarities.append(sim)

base_similarities = np.array(base_similarities)

# Separate positive and negative pairs
pos_sims = base_similarities[val_labels == 1.0]
neg_sims = base_similarities[val_labels == 0.0]

print(f"\n📊 BASE MODEL EVALUATION:")
print("="*60)
print(f"   Positive pairs (relevant):")
print(f"      Mean similarity: {pos_sims.mean():.4f}")
print(f"      Min/Max: {pos_sims.min():.4f} / {pos_sims.max():.4f}")
print(f"\n   Negative pairs (irrelevant):")
print(f"      Mean similarity: {neg_sims.mean():.4f}")
print(f"      Min/Max: {neg_sims.min():.4f} / {neg_sims.max():.4f}")
print(f"\n   Separation (higher is better): {(pos_sims.mean() - neg_sims.mean()):.4f}")
print("="*60)

# Save for later comparison
base_pos_mean = pos_sims.mean()
base_neg_mean = neg_sims.mean()
base_separation = base_pos_mean - base_neg_mean

## Bước 10: Fine-tune Model 🔥

**Training parameters:**
- Loss: CosineSimilarityLoss (contrastive learning)
- Epochs: 3-5 (avoid overfitting)
- Warmup steps: 10% of training steps
- Evaluation: Every epoch

**Estimated time:**
- With T4 GPU: 5-10 minutes
- With CPU: 30-60 minutes

⚠️ **Quan trọng:** Đừng tắt notebook trong khi training!

In [ ]:
# Disable W&B tracking to avoid API key prompt
import os
os.environ['WANDB_DISABLED'] = 'true'

# Configure training
EPOCHS = 3
WARMUP_STEPS = int(len(train_dataloader) * 0.1)  # 10% of training steps
OUTPUT_PATH = './models/embedding_finetuned'

# Create loss function
train_loss = losses.CosineSimilarityLoss(base_model)

print("🚀 Starting fine-tuning...")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Training examples: {len(train_examples)}")
print(f"   Steps per epoch: {len(train_dataloader)}")
print(f"   Warmup steps: {WARMUP_STEPS}")
print(f"   Total training steps: {len(train_dataloader) * EPOCHS}")
print(f"\n💾 Model will be saved to: {OUTPUT_PATH}")
print("\n" + "="*60)

# Fine-tune (without evaluator - will evaluate manually after training)
base_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUTPUT_PATH,
    save_best_model=True,
    show_progress_bar=True,
    use_amp=False  # Disable automatic mixed precision to avoid compatibility issues
)

print("\n" + "="*60)
print("✅ Fine-tuning completed!")
print(f"\n💾 Model saved to: {OUTPUT_PATH}")

## Bước 11: Load Fine-tuned Model & Compare Performance

In [ ]:
# Load fine-tuned model from disk
print("📥 Loading fine-tuned model from saved checkpoint...")

try:
    finetuned_model = SentenceTransformer(OUTPUT_PATH)
    print("✅ Fine-tuned model loaded successfully!")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("   Falling back to base_model (which was trained in-place)")
    finetuned_model = base_model

print(f"   Embedding dimension: {finetuned_model.get_sentence_embedding_dimension()}")

# Evaluate fine-tuned model with SAME validation data
print("\n🔍 Evaluating fine-tuned model...")
print(f"   Encoding {len(val_query_texts)} validation pairs...")

finetuned_query_embeddings = finetuned_model.encode(val_query_texts, show_progress_bar=True)
finetuned_doc_embeddings = finetuned_model.encode(val_doc_texts, show_progress_bar=True)

# Calculate cosine similarities
finetuned_similarities = []
for i in range(len(finetuned_query_embeddings)):
    sim = cos_sim([finetuned_query_embeddings[i]], [finetuned_doc_embeddings[i]])[0][0]
    finetuned_similarities.append(sim)

finetuned_similarities = np.array(finetuned_similarities)

# Separate positive and negative pairs
ft_pos_sims = finetuned_similarities[val_labels == 1.0]
ft_neg_sims = finetuned_similarities[val_labels == 0.0]

print(f"\n📊 FINE-TUNED MODEL EVALUATION:")
print("="*60)
print(f"   Positive pairs (relevant):")
print(f"      Mean similarity: {ft_pos_sims.mean():.4f}")
print(f"      Min/Max: {ft_pos_sims.min():.4f} / {ft_pos_sims.max():.4f}")
print(f"\n   Negative pairs (irrelevant):")
print(f"      Mean similarity: {ft_neg_sims.mean():.4f}")
print(f"      Min/Max: {ft_neg_sims.min():.4f} / {ft_neg_sims.max():.4f}")
print(f"\n   Separation (higher is better): {(ft_pos_sims.mean() - ft_neg_sims.mean()):.4f}")
print("="*60)

# Calculate improvement
ft_pos_mean = ft_pos_sims.mean()
ft_neg_mean = ft_neg_sims.mean()
ft_separation = ft_pos_mean - ft_neg_mean

print(f"\n🎯 COMPARISON:")
print("="*60)
print(f"   Metric: Positive-Negative Separation")
print(f"\n   Base Model:       {base_separation:.4f}")
print(f"   Fine-tuned Model: {ft_separation:.4f}")
print(f"   Improvement:      {(ft_separation - base_separation):.4f} ({((ft_separation - base_separation)/base_separation*100):+.1f}%)")
print("="*60)

if ft_separation > base_separation:
    print("\n🎉 FINE-TUNING SUCCESSFUL! Model improved at distinguishing relevant vs irrelevant docs.")
else:
    print("\n⚠️ No improvement detected. Possible reasons:")
    print("   - Not enough training data (need 100+ docs per category)")
    print("   - Data quality issues (English instead of Vietnamese?)")
    print("   - More training epochs needed")
    print("   - Learning rate too high/low")

## Bước 12: Test với Real-world Queries

In [ ]:
# Test queries (Vietnamese health + weather)
test_queries = [
    "thời tiết nóng ảnh hưởng đến sức khỏe như thế nào",
    "UV cao có nguy hiểm không",
    "pm2.5 ảnh hưởng đến hô hấp",
    "dị ứng thời tiết",
    "tim mạch và nhiệt độ",
    "tiểu đường cần chú ý gì khi trời nóng",
    "hen suyễn khi ô nhiễm không khí",
    "triệu chứng say nắng"
]

# Get sample documents from each category
test_docs = []
for category in corpus_df['category'].unique():
    sample = corpus_df[corpus_df['category'] == category].sample(2)
    test_docs.extend(sample['text'].tolist())

print("🧪 Testing both models with real queries...")
print(f"   Test queries: {len(test_queries)}")
print(f"   Test documents: {len(test_docs)}")
print("\n" + "="*80)

# IMPORTANT: Need to reload base model because it was trained in-place!
print("\n⚠️ Note: Reloading base model to compare properly...")
base_model_original = SentenceTransformer(MODEL_NAME)  # Fresh untrained model
print("   ✅ Fresh base model loaded")

# Encode with both models
print("\n📥 Encoding with base model...")
query_embeddings_base = base_model_original.encode(test_queries)
doc_embeddings_base = base_model_original.encode(test_docs)

print("📥 Encoding with fine-tuned model...")
query_embeddings_finetuned = finetuned_model.encode(test_queries)
doc_embeddings_finetuned = finetuned_model.encode(test_docs)

# Calculate similarities
similarities_base = cosine_similarity(query_embeddings_base, doc_embeddings_base)
similarities_finetuned = cosine_similarity(query_embeddings_finetuned, doc_embeddings_finetuned)

# Show results for first 3 queries
for i in range(min(3, len(test_queries))):
    query = test_queries[i]
    
    # Top-3 docs for base model
    top_base_idx = similarities_base[i].argsort()[-3:][::-1]
    
    # Top-3 docs for finetuned model
    top_finetuned_idx = similarities_finetuned[i].argsort()[-3:][::-1]
    
    print(f"\n🔍 Query {i+1}: {query}")
    print("\n📘 Base Model - Top 3:")
    for rank, idx in enumerate(top_base_idx, 1):
        print(f"   {rank}. Score: {similarities_base[i][idx]:.4f}")
        print(f"      Doc: {test_docs[idx][:100]}...")
    
    print("\n📗 Fine-tuned Model - Top 3:")
    for rank, idx in enumerate(top_finetuned_idx, 1):
        print(f"   {rank}. Score: {similarities_finetuned[i][idx]:.4f}")
        print(f"      Doc: {test_docs[idx][:100]}...")
    
    print("\n" + "-"*80)

print("\n✅ Test completed!")

## Bước 13: Visualization - Performance Comparison

In [ ]:
# Plot similarity score distributions
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Base model
axes[0].hist(similarities_base.flatten(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Base Model - Similarity Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cosine Similarity', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(similarities_base.flatten().mean(), color='red', linestyle='--', label=f'Mean: {similarities_base.flatten().mean():.3f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Fine-tuned model
axes[1].hist(similarities_finetuned.flatten(), bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title('Fine-tuned Model - Similarity Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Cosine Similarity', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].axvline(similarities_finetuned.flatten().mean(), color='red', linestyle='--', label=f'Mean: {similarities_finetuned.flatten().mean():.3f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./similarity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved: ./similarity_comparison.png")

# Statistics
print(f"\n📊 Similarity Statistics:")
print(f"\nBase Model:")
print(f"   Mean: {similarities_base.flatten().mean():.4f}")
print(f"   Std:  {similarities_base.flatten().std():.4f}")
print(f"   Max:  {similarities_base.flatten().max():.4f}")

print(f"\nFine-tuned Model:")
print(f"   Mean: {similarities_finetuned.flatten().mean():.4f}")
print(f"   Std:  {similarities_finetuned.flatten().std():.4f}")
print(f"   Max:  {similarities_finetuned.flatten().max():.4f}")

## 🔽 Bước 14: Download Fine-tuned Model

**Files cần download:**
- Toàn bộ folder `./models/embedding_finetuned/` (~470 MB)

**Nơi lưu:**
- Local: `E:\CĐTT2\WeatherWell AI\models\embedding_finetuned\`

**Next steps:**
1. Download model về máy
2. Update `core/langchain_local_adapter.py` để load model mới
3. Re-index ChromaDB với embedding mới
4. Test RAG pipeline với model mới

In [ ]:
# Compress model folder for download
import shutil

print("📦 Compressing model folder...")
shutil.make_archive(
    './embedding_finetuned',  # Output filename (without extension)
    'zip',                    # Archive format
    './models',               # Root directory
    'embedding_finetuned'     # Folder to compress
)

zip_size = os.path.getsize('./embedding_finetuned.zip') / (1024*1024)
print(f"\n✅ Model compressed: embedding_finetuned.zip ({zip_size:.1f} MB)")

# Download
print("\n📥 Downloading fine-tuned model...")
print("   Browser download will start...\n")

files.download('./embedding_finetuned.zip')

print("\n✅ Download started!")
print("\n📋 Next steps:")
print("   1. Extract embedding_finetuned.zip")
print("   2. Move to: E:\\CĐTT2\\WeatherWell AI\\models\\embedding_finetuned\\")
print("   3. Update code to use new model")
print("   4. Re-index ChromaDB")
print("\n💡 See integration guide in next cell")

## 📚 Integration Guide

### **1. Update `core/langchain_local_adapter.py`**

```python
# OLD:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# NEW:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('./models/embedding_finetuned')  # Local path
```

---

### **2. Re-index ChromaDB**

Chạy lại notebook `02_push_to_chromadb_with_gemini.ipynb` hoặc script:

```python
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Create new embedding function
finetuned_ef = SentenceTransformerEmbeddingFunction(
    model_name='./models/embedding_finetuned'
)

# Create new collection
client = chromadb.PersistentClient(path='./chroma_data')
collection = client.create_collection(
    name='health_corpus_v2',  # New collection name
    embedding_function=finetuned_ef
)

# Re-add all documents
for doc in corpus:
    collection.add(
        documents=[doc['text']],
        metadatas=[doc['metadata']],
        ids=[doc['id']]
    )
```

---

### **3. Test RAG Pipeline**

```python
# Test query
query = "thời tiết nóng ảnh hưởng đến sức khỏe như thế nào"

# Query with fine-tuned embeddings
results = collection.query(
    query_texts=[query],
    n_results=5
)

print("Top 5 documents:")
for i, doc in enumerate(results['documents'][0], 1):
    print(f"{i}. {doc[:150]}...")
    print(f"   Distance: {results['distances'][0][i-1]:.4f}\n")
```

---

### **4. Compare Performance (Optional)**

A/B test với users:
- 50% users → old embedding
- 50% users → fine-tuned embedding
- Measure: answer relevance, user satisfaction

---

## ✅ Checklist

- [ ] Download `embedding_finetuned.zip`
- [ ] Extract to `./models/embedding_finetuned/`
- [ ] Update `langchain_local_adapter.py`
- [ ] Re-index ChromaDB
- [ ] Test with sample queries
- [ ] Deploy to production
- [ ] Monitor performance improvements

---

## 🎉 Congratulations!

Bạn đã fine-tune thành công embedding model cho domain-specific RAG system!

**Expected improvements:**
- Better semantic understanding of Vietnamese health queries
- More accurate document retrieval
- Improved RAG answer quality
- Lower false positive matches